In [ ]:
# ============================================================
# Notebook: Bayesian-style sampling using Logistic Regression
# ============================================================

# Imports
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression

# Load data
X = np.load("/mnt/data/initial_inputs.npy")      # shape (N,2)
y = np.load("/mnt/data/initial_outputs.npy")     # shape (N,)

# Convert outputs to binary (non-zero = contamination detected)
y_binary = (y > 0).astype(int)

# Train logistic regression
model = LogisticRegression()
model.fit(X, y_binary)

# Create candidate grid
x_min, x_max = X[:,0].min(), X[:,0].max()
y_min, y_max = X[:,1].min(), X[:,1].max()

gx = np.linspace(x_min, x_max, 100)
gy = np.linspace(y_min, y_max, 100)
XX, YY = np.meshgrid(gx, gy)

X_grid = np.vstack([XX.ravel(), YY.ravel()]).T

# Predict probabilities
probs = model.predict_proba(X_grid)[:,1]

# Acquisition (uncertainty + exploration proxy)
# choose points near decision boundary (prob ~ 0.5)
acquisition = -np.abs(probs - 0.5)

# Select next (10,2) inputs
top_idx = np.argsort(acquisition)[-10:]
next_points = X_grid[top_idx]

print("Next proposed inputs (10,2):")
print(next_points)

# Plot
plt.figure(figsize=(6,5))
plt.contourf(XX, YY, probs.reshape(XX.shape), levels=30)
plt.scatter(X[:,0], X[:,1], label="Existing samples")
plt.scatter(next_points[:,0], next_points[:,1], marker='x', s=100, label="Next samples")
plt.legend()
plt.show()